# 04 — Train the LSTM Caption Decoder on Google Colab (GPU)

This notebook trains the decoder from `src/train_lstm.py` on a Colab GPU, using the **VGG16 features we already extracted** (notebook 02). VGG16 is *not* run again.

```
saved VGG16 fc2 vector (4096) ─► Dropout ─► Dense(256, ReLU) ──────────┐
                                                                       ├─► Add ─► Dense(256, ReLU) ─► Dropout ─► Softmax(7,702 words)
caption prefix (39 word IDs) ─► Embedding(256) ─► Dropout ─► LSTM(256) ─┘                                     = next word
```

**Every epoch is saved to Google Drive.** If Colab disconnects, reconnect and run all cells again: training continues from the last finished epoch.

### Before you start (one time)
1. In Colab: **Runtime → Change runtime type → T4 GPU**.
2. In Google Drive, create `MyDrive/NIT3251_captioning/features/` and upload these 4 files from your computer's `data/features/` folder (≈133 MB):
   `features.npy`, `image_ids.json`, `progress.npy`, `metadata.json`.
3. Leave `RUN_TRAINING = False` for the first run to check everything works, then set it to `True`.

The same notebook also runs on a normal computer (it then uses the project's `data/` and `models/` folders instead of Drive).

## 1. Settings — the only cell you normally edit

In [ ]:
# Where things live on Google Drive (Colab only).
DRIVE_PROJECT = "/content/drive/MyDrive/NIT3251_captioning"
RUN_NAME = "lstm_run1"            # use a new name for each experiment, e.g. "lstm_run2_bs256"

# Code to use. Change to "main" once the training PR has been merged.
REPO_URL = "https://github.com/JustinDang69/Automatic-Image-Captioning-with-CNN-and-LSTM-.git"
REPO_BRANCH = "feature/lstm-training-colab"

# Training hyper-parameters (architecture sizes are fixed at 256 in build_caption_model).
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 0.001
DROPOUT = 0.4
PATIENCE = 4                      # stop after 4 epochs without val_loss improvement

SMOKE_TEST = True                 # quick 2-epoch run on 50 images to test GPU + Drive saving
RUN_TRAINING = False              # set True to train on all 6,472 training images

## 2. Environment: Colab or local computer?

In Colab we mount Google Drive and download the code from GitHub. Locally we use the project folder that this notebook is in.

In [ ]:
import os, sys, shutil, subprocess, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_ROOT = Path("/content/nit3251-captioning")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(PROJECT_ROOT)], check=True)
    else:   # re-run after a disconnect: update the existing copy to REPO_BRANCH
        for cmd in (["fetch", "origin"], ["checkout", REPO_BRANCH], ["pull", "--ff-only"]):
            subprocess.run(["git", "-C", str(PROJECT_ROOT), *cmd], check=True)
    DRIVE_DIR = Path(DRIVE_PROJECT)
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE_DIR = PROJECT_ROOT / "models" / "colab_runs"     # stands in for Google Drive

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

DRIVE_FEATURES = DRIVE_DIR / "features"          # Colab: your uploaded features
RUN_DIR = DRIVE_DIR / "runs" / RUN_NAME            # checkpoints, history, graph (persistent)
print("Running in Colab:", IN_COLAB)
print("Project code    :", PROJECT_ROOT)
print("Checkpoints to  :", RUN_DIR)

In [ ]:
import numpy as np
import tensorflow as tf
import keras

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
assert int(keras.__version__.split(".")[0]) >= 3, "Keras 3 is required (pip install -U keras)"
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE - training will be very slow. In Colab: Runtime > Change runtime type > T4 GPU")

## 3. Flickr8k captions

The decoder needs the **captions** (`captions.txt`). `prepare_flickr8k` also checks that every image file exists, so in Colab we download the Kaggle dataset (≈1 GB, about a minute) with `kagglehub`. Locally the existing `data/flickr8k/` folder is used.

If the Kaggle download fails in Colab, upload `captions.txt` and the `Images/` folder to `MyDrive/NIT3251_captioning/flickr8k/` and the cell will use that instead.

In [ ]:
if IN_COLAB:
    drive_copy = DRIVE_DIR / "flickr8k"
    if (drive_copy / "captions.txt").exists():
        FLICKR8K_DIR = drive_copy
    else:
        import kagglehub
        FLICKR8K_DIR = Path(kagglehub.dataset_download("adityajn105/flickr8k"))
else:
    FLICKR8K_DIR = PROJECT_ROOT / "data" / "flickr8k"

os.environ["FLICKR8K_DIR"] = str(FLICKR8K_DIR)   # read by src.data_preprocessing
print("Flickr8k folder:", FLICKR8K_DIR, "| contains:", sorted(p.name for p in FLICKR8K_DIR.iterdir()))

## 4. Load the saved VGG16 features (no re-extraction)

`features.npy` holds one 4,096-number vector per image, row `i` belonging to `image_ids.json[i]`; `progress.npy` confirms every row was written. In Colab we copy them from Drive to the fast local disk first. **This notebook never runs VGG16** — if the files are missing it stops with instructions.

In [ ]:
from src.feature_extraction import load_features, FEATURE_DIM

FEATURE_FILES = ["features.npy", "image_ids.json", "progress.npy", "metadata.json"]
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

if IN_COLAB:
    missing = [f for f in FEATURE_FILES if not (DRIVE_FEATURES / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing {missing} in {DRIVE_FEATURES}. Upload them from your computer's data/features/ folder "
            "(created by notebook 02). Do not re-extract VGG16 features in Colab.")
    FEATURES_DIR.mkdir(parents=True, exist_ok=True)
    for f in FEATURE_FILES:
        shutil.copy2(DRIVE_FEATURES / f, FEATURES_DIR / f)

features, row_of = load_features(FEATURES_DIR, require_complete=True)   # stops if extraction was incomplete
meta = json.loads((FEATURES_DIR / "metadata.json").read_text())
print("features:", features.shape, features.dtype, "| layer:", meta["layer"], "| model:", meta["model"])
assert features.shape == (len(row_of), FEATURE_DIM) and meta["complete"]

## 5. How the pieces fit together (read this before the Q&A)

### 5.1 `prepare_flickr8k()` — `src/data_preprocessing.py`
* **Input:** the Flickr8k folder.
* **Does:** cleans captions (`startseq a dog runs endseq`), splits **images** 80/10/10 with seed 42, fits the tokenizer on **training captions only**.
* **Output:** a `PreparedData` object with `captions`, `splits`, `tokenizer`, `vocab_size` (7,702) and `max_length` (39).

In [ ]:
from src.data_preprocessing import prepare_flickr8k

prepared = prepare_flickr8k(seed=42)
print({k: len(v) for k, v in prepared.splits.items()}, "| vocab_size:", prepared.vocab_size,
      "| max_length:", prepared.max_length)
assert all(i in row_of for ids in prepared.splits.values() for i in ids), "every image needs a feature row"
example_id = prepared.splits["train"][0]
print("example:", example_id, "->", prepared.captions[example_id][0])

### 5.2 `prepare_batches()` and `CaptionBatches` — `src/train_lstm.py`
* **Input:** one split (`"train"` or `"val"` — `"test"` is refused), the features and the tokenizer.
* **Does:** `create_sequences` turns each caption into (prefix → next word) pairs. `CaptionBatches` then serves them in batches and, **only for the pairs in the current batch**, looks up each image's 4,096-d vector with `features[rows]`. Copying the vector for all 381,317 training pairs would need ≈6 GB.
* **Output per batch:** `{"image_features": (B, 4096), "caption_input": (B, 39)}` and `targets: (B,)` (word IDs).

Training batches are reshuffled every epoch; validation batches are not.

In [ ]:
from src.train_lstm import prepare_batches

train_batches, n_train = prepare_batches(prepared, features, row_of, "train", batch_size=BATCH_SIZE)
val_batches, n_val = prepare_batches(prepared, features, row_of, "val", batch_size=BATCH_SIZE)
print(f"train: {n_train:,} pairs = {len(train_batches):,} batches | val: {n_val:,} pairs = {len(val_batches):,} batches")

x, y = train_batches[0]
print("image_features:", x["image_features"].shape, "| caption_input:", x["caption_input"].shape, "| targets:", y.shape)
tok = prepared.tokenizer
for k in range(3):
    print(f"  '{tok.sequences_to_texts([x['caption_input'][k]])[0]}'  ->  '{tok.index_word[int(y[k])]}'")

### 5.3 `build_caption_model()` — the decoder
* **Inputs:** `image_features (None, 4096)` and `caption_input (None, 39)`.
* **Image branch:** Dropout → Dense(256, ReLU) projects VGG16's 4,096 numbers to 256.
* **Text branch:** Embedding turns each word ID into a 256-d vector → Dropout → LSTM(256) reads the prefix and outputs one 256-d summary.
* **Fusion:** `Add` sums the two 256-d vectors element-wise → Dense(256, ReLU) → Dropout → Dense(vocab_size, softmax).
* **Output:** `(None, 7702)` — a probability for every vocabulary word being the next word.
* **Loss:** sparse categorical cross-entropy (targets are word IDs, not one-hot). **Optimizer:** Adam.

**Why no padding mask?** Captions are *left*-padded with 0. On a GPU, Keras uses NVIDIA cuDNN for the LSTM, and cuDNN only accepts masks for *right*-padded data — with a mask, training would crash. Without a mask the LSTM simply learns to ignore the padding token, and because padding is on the left the real words are always the last thing it reads.

In [ ]:
from src.train_lstm import build_caption_model

model = build_caption_model(prepared.vocab_size, prepared.max_length,
                            learning_rate=LEARNING_RATE, dropout=DROPOUT)
print(f"{'layer':22s} {'output shape':22s} params")
for layer in model.layers:
    print(f"{layer.name:22s} {str(layer.output.shape):22s} {layer.count_params():,}")
print(f"Total trainable parameters: {model.count_params():,} (VGG16 is not included - it is frozen and pre-computed)")

### 5.4 GPU check: one real training step
Runs a few batches through the full-size model to confirm the GPU (cuDNN) path works and to estimate the time per epoch. The first step is slower because TensorFlow compiles the graph.

In [ ]:
model.train_on_batch(x, y)                       # warm-up / compile
t = time.perf_counter()
for b in range(1, 11):
    xb, yb = train_batches[b]
    logs = model.train_on_batch(xb, yb, return_dict=True)
step = (time.perf_counter() - t) / 10
print(f"loss after 11 steps: {logs['loss']:.3f} | {step * 1000:.0f} ms/step on {'GPU' if gpus else 'CPU'}")
print(f"estimated epoch time: {step * len(train_batches) / 60:.1f} min (+ validation)")
del model

## 6. Persistent checkpoints — what is saved to Drive

`train()` adds the `ResumableTraining` callback. After **every epoch** it writes to `RUN_DIR`:

| File | Purpose |
|---|---|
| `last_checkpoint.keras` | weights **and** optimizer state of the latest epoch → used to resume |
| `best_caption_model.keras` | model with the lowest validation loss so far |
| `caption_model.keras` | copy of the best model when training finishes (use this for decoding/BLEU) |
| `training_state.json` | epochs done, best epoch, early-stopping and learning-rate counters, history |
| `history.json`, `training_history.csv` | loss / accuracy / learning rate per epoch |
| `training_loss.png` | training vs validation loss graph for the report |
| `tokenizer.json`, `splits.json`, `metadata.json` | exact vocabulary and split used — needed for decoding |
| `training_config.json` | hyper-parameters and final summary |

Files are written to a temporary name and then renamed, so a disconnect mid-save cannot corrupt an existing checkpoint. On restart, `train()` loads `last_checkpoint.keras` and continues at the next epoch (it refuses to resume if the vocabulary or settings changed). Early stopping and learning-rate halving use **validation loss only**; the 810 test images are never used.

In [ ]:
from src.train_lstm import train

if SMOKE_TEST:
    smoke_dir = DRIVE_DIR / "runs" / "smoke_test"
    print("--- session 1: 1 epoch on 50 train / 50 val images")
    train(output_dir=smoke_dir, epochs=1, batch_size=BATCH_SIZE, max_images=50, resume=False, verbose=2)
    print("\n--- session 2: same command with epochs=2 -> resumes after epoch 1")
    _, smoke_history, _ = train(output_dir=smoke_dir, epochs=2, batch_size=BATCH_SIZE, max_images=50, verbose=2)
    assert len(smoke_history["loss"]) == 2
    print("\nSmoke test OK. Files on Drive:", sorted(p.name for p in smoke_dir.iterdir()))

## 7. Full training

Set `RUN_TRAINING = True` in the settings cell, then **Runtime → Run all**.

* Progress is printed once per epoch.
* If Colab disconnects or times out: reconnect, check the GPU runtime, **Run all** again — it resumes from `RUN_DIR`.
* To start a new experiment from scratch, change `RUN_NAME` (old runs are kept).

In [ ]:
if RUN_TRAINING:
    best_model, history, prepared = train(
        output_dir=RUN_DIR, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
        dropout=DROPOUT, patience=PATIENCE, seed=42, verbose=2,
    )
else:
    print("RUN_TRAINING is False - skipping full training.")

## 8. Results (from the files on Drive)

In [ ]:
from IPython.display import Image, display

if (RUN_DIR / "training_state.json").exists():
    state = json.loads((RUN_DIR / "training_state.json").read_text())
    h = state["history"]
    print(f"epochs completed: {state['epochs_completed']} | best epoch: {state['best_epoch']} "
          f"| best val_loss: {state['best_val_loss']:.4f} | stopped early: {state['stopped_early']}")
    print(f"{'epoch':>5} {'loss':>8} {'val_loss':>9} {'val_acc':>8} {'lr':>9}")
    for i in range(len(h["loss"])):
        print(f"{i + 1:5d} {h['loss'][i]:8.4f} {h['val_loss'][i]:9.4f} "
              f"{h['val_sparse_categorical_accuracy'][i]:8.4f} {h['learning_rate'][i]:9.2e}")
    display(Image(str(RUN_DIR / "training_loss.png")))
else:
    print("No training run found in", RUN_DIR, "yet.")

**Reading the graph:** training loss should keep falling; validation loss falls and then flattens or rises. The best epoch (lowest validation loss) is the model kept in `caption_model.keras`. Next-word accuracy is measured with the *true* previous words (teacher forcing), so it is **not** caption quality — that is measured later with greedy/beam search and BLEU on the test images.

## 9. Next steps
* Use `RUN_DIR/caption_model.keras` + `tokenizer.json` for greedy and beam-search decoding (beam widths 3 and 5).
* Evaluate BLEU-1…4 on the held-out test split (`splits.json["test"]`).